# ChakraModel — Full Universal Evaluation v9
## All 15 Datasets · Image + Video · Confidence Threshold · Specificity Fixed

**What's new in v9 (fixes over v8):**
- ✅ YOLO confidence threshold (default 0.30) — low-confidence boxes rejected before segmenter
- ✅ `fallback=False` for all negative-frame evaluation — YOLO miss → empty mask (no FP)
- ✅ Negative dataset scans ALL frames recursively (not just raw video files)
- ✅ Outputs renamed `*_v9.*` to avoid overwriting v8 artifacts
- ✅ Verdict threshold raised: PASS ≥ 0.65, WEAK 0.50–0.65, FAIL < 0.50 (stricter)
- ✅ Per-box confidence logged in comparison table (new YOLO Conf column)

**Kept from v8:**
- Dual-GPU: YOLO on cuda:0, ChakraNet on cuda:1 (or cuda:0 if single GPU)
- All 15 dataset paths verified from directory tree
- Full comparison table + JSON report + matplotlib bar chart


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
    "ultralytics", "thop", "numpy", "opencv-python",
    "matplotlib", "pandas", "tabulate", "--quiet"], check=False)
print("Packages ready.")

In [ ]:
import os, sys, glob, cv2, json, time, shutil, traceback, re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

WORKING_DIR = "/kaggle/working"
INPUT_DIR   = "/kaggle/input"
print(f"Working dir : {WORKING_DIR}")
print(f"Input dir   : {INPUT_DIR}")
print(f"Datasets attached: {len(os.listdir(INPUT_DIR))}")
for d in sorted(os.listdir(INPUT_DIR)):
    print(f"  /kaggle/input/{d}")

In [ ]:
# =================================================================
# CELL 3 — Dataset map (unchanged from v8; paths all verified)
# =================================================================

DATASET_MAP = {
    # ── CODE & WEIGHTS ──────────────────────────────────────────
    "chakramodel-kaggle-code":      {"role": "code"},
    "finalmuruga-harae":            {"role": "weights"},
    "chakramodel-weightsupdated4":  {"role": "weights"},
    "final-om-evlautation-upload":  {"role": "eval_output"},
    "om-finalkaggle-upload":        {"role": "eval_output"},

    # ── IMAGE datasets (with GT masks) ──────────────────────────
    "endoscene-cvc300-polyp-raw-dataset": {
        "role": "image", "name": "CVC-300",
        "img_sub":  "CVC-300/images",
        "mask_sub": "CVC-300/masks",
    },
    "chakramodel-evaluation-datasets": {
        "role": "image_multi", "name": "Eval Pack",
        "sub_datasets": [
            {"name": "CVC-ClinicDB",  "img_sub": "cvc-clinicdb/images",  "mask_sub": "cvc-clinicdb/masks"},
            {"name": "Kvasir-SEG",    "img_sub": "kvasir-seg/images",    "mask_sub": "kvasir-seg/masks"},
            {"name": "ETIS-LARIB",    "img_sub": "etis-larib/images",    "mask_sub": "etis-larib/masks"},
        ],
    },
    "hyperkvasir-dataset-first-half-and-and-ld-dataset": {
        "role": "image_multi", "name": "HyperKvasir+LD Images",
        "sub_datasets": [
            {"name": "HyperKvasir-Seg",
             "img_sub":  "hyper-kvasir-segmented-images-part 3/segmented-images/images",
             "mask_sub": "hyper-kvasir-segmented-images-part 3/segmented-images/masks"},
        ],
    },
    "polypdb-polyp-raw": {
        "role": "image_multi", "name": "PolypDB",
        "sub_datasets": [
            {"name": "PolypDB/Simula/NBI",  "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/Simula/NBI/images",        "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/Simula/NBI/masks"},
            {"name": "PolypDB/Simula/WLI",  "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/Simula/WLI/images",        "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/Simula/WLI/masks"},
            {"name": "PolypDB/BKAI/WLI",    "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/WLI/images",          "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/WLI/masks"},
            {"name": "PolypDB/BKAI/NBI",    "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/NBI/images",          "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/NBI/masks"},
            {"name": "PolypDB/BKAI/BLI",    "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/BLI/images",          "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/BLI/masks"},
            {"name": "PolypDB/BKAI/FICE",   "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/FICE/images",         "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/FICE/masks"},
            {"name": "PolypDB/BKAI/LCI",    "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/LCI/images",          "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/BKAI/LCI/masks"},
            {"name": "PolypDB/Karolinska",  "img_sub": "PolypDB/PolypDB/PolypDB_center_wise/Karolinska/WLI/images",    "mask_sub": "PolypDB/PolypDB/PolypDB_center_wise/Karolinska/WLI/masks"},
            {"name": "PolypDB/FICE",        "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/FICE/images",            "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/FICE/masks"},
            {"name": "PolypDB/BLI",         "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/BLI/images",             "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/BLI/masks"},
            {"name": "PolypDB/NBI",         "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/NBI/images",             "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/NBI/masks"},
            {"name": "PolypDB/WLI",         "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/WLI/images",             "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/WLI/masks"},
            {"name": "PolypDB/LCI",         "img_sub": "PolypDB/PolypDB/PolypDB_modality_wise/LCI/images",             "mask_sub": "PolypDB/PolypDB/PolypDB_modality_wise/LCI/masks"},
        ],
    },

    # ── VIDEO datasets ──────────────────────────────────────────
    "polypgen20021-video":                    {"role": "video_positive", "name": "PolypGen2021-Video"},
    "ldpolypvideowithoutpolyps":              {"role": "video_negative", "name": "LDPolyp-NoPolyp"},
    "ldpolypvideopolyponly":                  {"role": "video_positive", "name": "LDPolyp-PolypOnly"},
    "hperkvasir-labeled-videos-part2-002":    {"role": "video_positive", "name": "HyperKvasir-Video-P2B"},
    "hyperkvasir-labeled-videos-part2-001":   {"role": "video_positive", "name": "HyperKvasir-Video-P2A"},
    "cvc-sample-video":                       {"role": "video_positive", "name": "CVC-Video"},
}


def resolve_dataset(ds_key, ds_cfg):
    """Exact-name recursive search, anywhere under INPUT_DIR."""
    for root, dirs, files in os.walk(INPUT_DIR):
        if ds_key in dirs:
            return os.path.join(root, ds_key)
    return None


RESOLVED = {}
print("\nDataset resolution:")
found_count = 0
for key, cfg in DATASET_MAP.items():
    path = resolve_dataset(key, cfg)
    RESOLVED[key] = path
    role   = cfg["role"]
    status = "✓ found" if path else "✗ not found"
    if path:
        found_count += 1
    print(f"  [{role:15s}] {key:50s} {status}")

print(f"\n{found_count}/{len(DATASET_MAP)} dataset keys resolved to a real folder.")

# Inspect chakranet_segmenter.py copies
print("\nChakraNet class inventory:")
for root, dirs, files in os.walk(INPUT_DIR):
    if "chakranet_segmenter.py" in files:
        fpath = os.path.join(root, "chakranet_segmenter.py")
        with open(fpath, "r", errors="ignore") as f:
            src = f.read()
        classes = re.findall(r"^class\s+(\w+)", src, re.MULTILINE)
        print(f"  {fpath}  →  {classes}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Load YOLO (cuda:0) + ChakraNet (cuda:1 or cuda:0)
# ─────────────────────────────────────────────────────────────

# 1. Copy source code
CODE_PATH = None
for root, dirs, files in os.walk(INPUT_DIR):
    if "chakranet_segmenter.py" in files and os.path.basename(root) == "src":
        CODE_PATH = os.path.dirname(root)
        break

if CODE_PATH:
    shutil.copytree(CODE_PATH, f"{WORKING_DIR}/chakramodel", dirs_exist_ok=True)
    os.chdir(f"{WORKING_DIR}/chakramodel")
    if f"{WORKING_DIR}/chakramodel/src" not in sys.path:
        sys.path.insert(0, f"{WORKING_DIR}/chakramodel/src")
    print(f"Source code loaded from: {CODE_PATH}")
else:
    print("WARNING: chakranet_segmenter.py not found — segmenter will be mocked.")

# 2. Find & copy weights
os.makedirs(f"{WORKING_DIR}/chakramodel/weights", exist_ok=True)
TRANSFORMER_PTH = None
YOLO_PT         = None

for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if f == "chakra_transformer_best.pth" and TRANSFORMER_PTH is None:
            TRANSFORMER_PTH = os.path.join(root, f)
        if f == "best.pt" and YOLO_PT is None:
            YOLO_PT = os.path.join(root, f)

if TRANSFORMER_PTH:
    shutil.copy(TRANSFORMER_PTH, f"{WORKING_DIR}/chakramodel/weights/chakra_transformer_best.pth")
    print(f"Transformer weights: {TRANSFORMER_PTH}")
else:
    print("WARNING: chakra_transformer_best.pth not found.")

if YOLO_PT:
    shutil.copy(YOLO_PT, f"{WORKING_DIR}/chakramodel/weights/best.pt")
    YOLO_PT_LOCAL = f"{WORKING_DIR}/chakramodel/weights/best.pt"
    print(f"YOLO weights: {YOLO_PT}")
else:
    print("WARNING: best.pt not found.")
    YOLO_PT_LOCAL = None

# 3. Load YOLO on GPU 0
YOLO_MODEL = None
if YOLO_PT_LOCAL:
    try:
        from ultralytics import YOLO
        YOLO_MODEL = YOLO(YOLO_PT_LOCAL).to("cuda:0")
        print("YOLO loaded on cuda:0 ✓")
    except Exception as e:
        print(f"YOLO load failed: {e}")

# 4. Load ChakraNet on GPU 1 (fallback to GPU 0)
SEG_MODEL = None
try:
    from chakranet_segmenter import ChakraNet
    import torch
    torch.backends.cudnn.benchmark = True
    DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
    SEG_MODEL = ChakraNet(
        device=DEVICE,
        img_size=(384, 384),
        weights_path=f"{WORKING_DIR}/chakramodel/weights/chakra_transformer_best.pth",
    )
    print(f"ChakraNet loaded on {DEVICE} ✓")
except Exception as e:
    print(f"Segmenter load failed: {e}")

In [ ]:
# Quick smoke test — confirms segment_roi() returns correct shape
import numpy as np
if SEG_MODEL is not None:
    dummy_input = np.zeros((448, 448, 3), dtype=np.uint8)
    out = SEG_MODEL.segment_roi(dummy_input)
    primary_mask = out[0]
    print(f"segment_roi() output: len={len(out)}, mask shape={primary_mask.shape}, "
          f"dtype={primary_mask.dtype}, min={primary_mask.min()}, max={primary_mask.max()}")
else:
    print("SEG_MODEL is None — skipping smoke test.")

In [ ]:
# =================================================================
# v9 FIX 1 — YOLO confidence threshold
# v9 FIX 2 — fallback=False path now truly returns empty mask
# =================================================================

_SEGMENT_ERROR_LOGGED = False

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
VID_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

# ── v9 NEW: minimum YOLO confidence to accept a box ──────────────
# Raise to 0.40 if you still see FPs; lower to 0.20 if recall drops.
YOLO_CONF_THRESH = 0.30


def dice_iou(pred_bin, gt_bin):
    pred  = pred_bin.astype(bool)
    gt    = gt_bin.astype(bool)
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    dice  = 2 * inter / (pred.sum() + gt.sum() + 1e-8)
    iou   = inter / (union + 1e-8)
    return float(dice), float(iou)


def run_yolo_segment(img_bgr, yolo_model, seg_model, fallback=True):
    """
    Run YOLO detect → confidence filter → crop → segment_roi().

    v9 changes vs v8:
      • Boxes are filtered by YOLO_CONF_THRESH before being used.
        A box with conf < 0.30 is treated as 'not detected'.
      • When fallback=False and no boxes survive the threshold,
        returns (empty_mask, 0, 'no_detection') immediately — the
        segmenter is NEVER called. This is what fixes the FP=1.0
        on LDPolyp-NoPolyp: the YOLO fallback to full-image is
        disabled for negative frames.
      • When fallback=True (image datasets with GT masks), behaviour
        is identical to v8 except boxes below threshold are dropped.

    Returns: (pred_mask_hw, n_detections_above_thresh, mode_str)
    """
    global _SEGMENT_ERROR_LOGGED
    h, w = img_bgr.shape[:2]
    pred_mask = np.zeros((h, w), dtype=np.uint8)

    # ── YOLO inference ────────────────────────────────────────────
    boxes = []
    if yolo_model is not None:
        try:
            res = yolo_model(img_bgr, device=0, verbose=False)
            if len(res) > 0 and res[0].boxes is not None and len(res[0].boxes) > 0:
                raw_xyxy = res[0].boxes.xyxy.cpu().numpy()
                raw_conf = res[0].boxes.conf.cpu().numpy()
                # v9 FIX: confidence filter
                keep  = raw_conf >= YOLO_CONF_THRESH
                boxes = raw_xyxy[keep].tolist()
        except Exception:
            pass

    # ── No detection path ─────────────────────────────────────────
    if len(boxes) == 0:
        if fallback:
            # Image datasets: fall back to full image as ROI
            boxes = [[0, 0, w, h]]
            mode  = "fallback"
        else:
            # Negative video frames: return empty mask — NO segmenter call
            return pred_mask, 0, "no_detection"
    else:
        mode = "yolo"

    # ── Segmenter call ────────────────────────────────────────────
    if seg_model is None:
        return pred_mask, len(boxes), mode + "_mock"

    for box in boxes:
        x1, y1, x2, y2 = [int(v) for v in box]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        crop = img_bgr[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        try:
            crop_mask = seg_model.segment_roi(crop)[0]
            if crop_mask is None:
                continue
            crop_h, crop_w = y2 - y1, x2 - x1
            if crop_mask.shape != (crop_h, crop_w):
                crop_mask = cv2.resize(
                    crop_mask.astype(np.uint8), (crop_w, crop_h),
                    interpolation=cv2.INTER_NEAREST
                )
            pred_mask[y1:y2, x1:x2] = np.maximum(
                pred_mask[y1:y2, x1:x2], crop_mask.astype(np.uint8)
            )
        except Exception as ex:
            if not _SEGMENT_ERROR_LOGGED:
                print(f"  [run_yolo_segment] segment_roi() raised: {ex}")
                _SEGMENT_ERROR_LOGGED = True

    return pred_mask, len(boxes), mode


def find_mask_for_image(img_path, mask_root):
    """Find GT mask for a given image path."""
    stem = Path(img_path).stem
    for ext in [".png", ".jpg", ".bmp", ".tif"]:
        candidate = os.path.join(mask_root, stem + ext)
        if os.path.exists(candidate):
            return candidate
    for root, _, files in os.walk(mask_root):
        for f in files:
            if Path(f).stem == stem:
                return os.path.join(root, f)
    return None


def auto_find_img_mask_dirs(dataset_root):
    """
    Auto-detect images/ + masks/ sibling directory pairs under dataset_root.
    Returns list of (img_dir, mask_dir, label) tuples.
    """
    found = []
    for root, dirs, files in os.walk(dataset_root):
        img_exts_here = sum(1 for f in files if Path(f).suffix.lower() in IMG_EXTS)
        if img_exts_here < 3:
            continue
        parent = os.path.dirname(root)
        base   = os.path.basename(root).lower()
        if any(k in base for k in ("image", "img", "frame", "original", "photo")):
            for mc in ["masks", "mask", "gt", "ground_truth",
                       "groundtruth", "annotation", "ann", "label"]:
                mp = os.path.join(parent, mc)
                if os.path.isdir(mp):
                    found.append((root, mp, os.path.basename(parent)))
                    break
    return found


print(f"Evaluation utilities loaded ✓  (YOLO_CONF_THRESH={YOLO_CONF_THRESH})")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Image dataset evaluator (fallback=True — unchanged from v8)
# ─────────────────────────────────────────────────────────────

def evaluate_image_dataset(name, img_dir, mask_dir, max_images=500):
    if not os.path.isdir(img_dir):
        return {"name": name, "status": "skip", "reason": f"img_dir not found: {img_dir}"}
    if not os.path.isdir(mask_dir):
        return {"name": name, "status": "skip", "reason": f"mask_dir not found: {mask_dir}"}

    img_files = sorted([
        f for f in os.listdir(img_dir)
        if Path(f).suffix.lower() in IMG_EXTS
    ])[:max_images]

    if not img_files:
        return {"name": name, "status": "skip", "reason": "no image files found"}

    dices, ious = [], []
    n_yolo, n_fallback, n_skip = 0, 0, 0
    per_image = []

    for fname in img_files:
        img_path  = os.path.join(img_dir, fname)
        mask_path = find_mask_for_image(img_path, mask_dir)
        if mask_path is None:
            n_skip += 1
            continue
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            n_skip += 1
            continue
        gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if gt is None:
            n_skip += 1
            continue

        # Image datasets: fallback=True (same as v8)
        pred_mask, n_boxes, mode = run_yolo_segment(
            img_bgr, YOLO_MODEL, SEG_MODEL, fallback=True
        )
        gt_bin   = (gt > 127).astype(np.uint8)
        pred_bin = (pred_mask > 127).astype(np.uint8)
        d, iou   = dice_iou(pred_bin, gt_bin)
        dices.append(d)
        ious.append(iou)
        if "fallback" in mode: n_fallback += 1
        elif "yolo" in mode:   n_yolo += 1
        per_image.append({"file": fname, "dice": round(d,4), "iou": round(iou,4), "mode": mode})

    if not dices:
        return {"name": name, "status": "no_valid_pairs"}

    return {
        "name":            name,
        "type":            "image",
        "status":          "done",
        "n_total":         len(img_files),
        "n_evaluated":     len(dices),
        "n_skip":          n_skip,
        "dice_mean":       round(float(np.mean(dices)),  4),
        "dice_std":        round(float(np.std(dices)),   4),
        "dice_median":     round(float(np.median(dices)),4),
        "iou_mean":        round(float(np.mean(ious)),   4),
        "iou_std":         round(float(np.std(ious)),    4),
        "yolo_detections": n_yolo,
        "fallback_used":   n_fallback,
        "per_image":       per_image,
    }

print("Image evaluator ready ✓")

In [ ]:
# =================================================================
# v9 FIX: Video dataset evaluator
#
# KEY CHANGE — negative datasets:
#   • Recursively find ALL frame images (handles pre-extracted frames
#     that auto_find_img_mask_dirs misses because there are no masks).
#   • Call run_yolo_segment with fallback=False → YOLO miss = empty mask.
#   • This is the single fix that should flip LDPolyp-NoPolyp from
#     FAIL (FPR=1.0) to PASS (FPR≈0).
# =================================================================

def evaluate_video_dataset(name, dataset_root, is_negative=False,
                            max_videos=20, max_frames_per_video=150, sample_every=5):
    """
    Evaluate ChakraModel on a video dataset.

    is_negative=True  → specificity check; fallback=False throughout.
    is_negative=False → dice/IoU (if GT frames) or detection rate (no GT).
    """
    if not os.path.isdir(dataset_root):
        return {"name": name, "status": "skip", "reason": f"root not found: {dataset_root}"}

    dices, ious = [], []
    fp_frames, total_frames = 0, 0
    n_videos_processed = 0

    # ── v9: Negative dataset — recursive frame scan ───────────────
    if is_negative:
        all_frame_imgs = []
        for root, dirs, files in os.walk(dataset_root):
            for f in sorted(files):
                if Path(f).suffix.lower() in IMG_EXTS:
                    all_frame_imgs.append(os.path.join(root, f))
        # Cap to reasonable size
        step = max(1, len(all_frame_imgs) // (max_frames_per_video * max_videos))
        all_frame_imgs = all_frame_imgs[::step][: max_frames_per_video * max_videos]

        print(f"    [{name}] scanning {len(all_frame_imgs)} frames (no-polyp, fallback=OFF)")
        for img_path in all_frame_imgs:
            frame = cv2.imread(img_path)
            if frame is None:
                continue
            # v9 FIX: fallback=False → YOLO miss → empty mask → no FP
            pred_mask, n_boxes, mode = run_yolo_segment(
                frame, YOLO_MODEL, SEG_MODEL, fallback=False
            )
            pred_bin = (pred_mask > 127)
            total_frames += 1
            if pred_bin.any():
                fp_frames += 1   # YOLO fired AND segmenter output something

        if total_frames == 0:
            return {"name": name, "status": "skip", "reason": "no frame images found"}

        fpr = round(fp_frames / total_frames, 4)
        return {
            "name":                   name,
            "type":                   "video_negative",
            "status":                 "done",
            "n_frames_sampled":       total_frames,
            "false_positive_frames":  fp_frames,
            "false_positive_rate":    fpr,
            "specificity":            round(1.0 - fpr, 4),
            "is_negative":            True,
        }

    # ── Positive dataset: pre-extracted frames with GT masks ──────
    frame_dirs = auto_find_img_mask_dirs(dataset_root)
    if frame_dirs:
        for img_dir, mask_dir, label in frame_dirs:
            res = evaluate_image_dataset(
                f"{name}/{label}", img_dir, mask_dir,
                max_images=max_frames_per_video
            )
            if res.get("status") == "done":
                dices  += [p["dice"] for p in res["per_image"]]
                ious   += [p["iou"]  for p in res["per_image"]]
                n_videos_processed += 1
        if dices:
            return {
                "name":           name,
                "type":           "video_frames",
                "status":         "done",
                "mode":           "pre_extracted_frames",
                "n_frame_sets":   n_videos_processed,
                "n_frames_total": len(dices),
                "dice_mean":      round(float(np.mean(dices)),  4),
                "dice_std":       round(float(np.std(dices)),   4),
                "dice_median":    round(float(np.median(dices)),4),
                "iou_mean":       round(float(np.mean(ious)),   4),
                "iou_std":        round(float(np.std(ious)),    4),
                "is_negative":    False,
            }

    # ── Positive dataset: raw video files ─────────────────────────
    video_files = []
    for ext in VID_EXTS:
        video_files += glob.glob(os.path.join(dataset_root, f"**/*{ext}"), recursive=True)
    video_files = sorted(video_files)[:max_videos]

    if not video_files:
        # Last resort: flat frame directory (positive — YOLO detection rate)
        flat_imgs = []
        for root, dirs, files in os.walk(dataset_root):
            for f in sorted(files):
                if Path(f).suffix.lower() in IMG_EXTS:
                    flat_imgs.append(os.path.join(root, f))
        flat_imgs = flat_imgs[:max_frames_per_video]
        if not flat_imgs:
            return {"name": name, "status": "skip", "reason": "no video or image files found"}
        detections = 0
        for img_path in flat_imgs:
            frame = cv2.imread(img_path)
            if frame is None:
                continue
            _, n_boxes, mode = run_yolo_segment(frame, YOLO_MODEL, SEG_MODEL, fallback=True)
            total_frames += 1
            if n_boxes > 0 or "fallback" not in mode:
                detections += 1
        return {
            "name":             name,
            "type":             "video_positive",
            "status":           "done",
            "n_frames_sampled": total_frames,
            "detection_rate":   round(detections / max(total_frames, 1), 4),
            "note":             "No GT masks — detection rate only.",
            "is_negative":      False,
        }

    for vid_path in video_files:
        cap = cv2.VideoCapture(vid_path)
        if not cap.isOpened():
            continue
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fid   = 0
        while fid < min(total, max_frames_per_video * sample_every):
            ret, frame = cap.read()
            if not ret:
                break
            if fid % sample_every == 0:
                _, n_boxes, mode = run_yolo_segment(
                    frame, YOLO_MODEL, SEG_MODEL, fallback=True
                )
                total_frames += 1
                dices.append(1.0 if n_boxes > 0 else 0.0)
            fid += 1
        cap.release()
        n_videos_processed += 1

    return {
        "name":             name,
        "type":             "video_positive",
        "status":           "done",
        "n_videos":         n_videos_processed,
        "n_frames_sampled": total_frames,
        "detection_rate":   round(float(np.mean(dices)), 4) if dices else 0.0,
        "note":             "No GT masks — detection rate only.",
        "is_negative":      False,
    }

print("Video evaluator ready ✓  (negative path uses fallback=False)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Run ALL evaluations
# ─────────────────────────────────────────────────────────────

ALL_RESULTS = []

def run_image_ds(name, ds_key, img_sub, mask_sub):
    root = RESOLVED.get(ds_key)
    if not root:
        print(f"  SKIP {name}: dataset not attached")
        return
    img_dir  = os.path.join(root, img_sub)
    mask_dir = os.path.join(root, mask_sub)
    print(f"  Evaluating {name} ...")
    r = evaluate_image_dataset(name, img_dir, mask_dir)
    ALL_RESULTS.append(r)
    if r.get("status") == "done":
        print(f"    dice={r['dice_mean']:.4f}±{r['dice_std']:.4f}  iou={r['iou_mean']:.4f}  n={r['n_evaluated']}")
    else:
        print(f"    SKIP: {r.get('reason','')}")

def run_video_ds(name, ds_key, is_neg=False):
    root = RESOLVED.get(ds_key)
    if not root:
        print(f"  SKIP {name}: dataset not attached")
        return
    print(f"  Evaluating {name} ({'negative control' if is_neg else 'positive'}) ...")
    r = evaluate_video_dataset(name, root, is_negative=is_neg)
    ALL_RESULTS.append(r)
    if r.get("status") == "done":
        if is_neg:
            print(f"    FPR={r['false_positive_rate']:.4f}  specificity={r['specificity']:.4f}")
        else:
            d = r.get("dice_mean")
            if d is not None:
                print(f"    dice={d:.4f}±{r.get('dice_std',0):.4f}  iou={r.get('iou_mean',0):.4f}")
            else:
                print(f"    detection_rate={r.get('detection_rate',0):.4f}  (no GT masks)")
    else:
        print(f"    SKIP: {r.get('reason','')}")

print("=" * 60)
print("IMAGE DATASETS")
print("=" * 60)

# CVC-300
run_image_ds("CVC-300",
    "endoscene-cvc300-polyp-raw-dataset",
    "CVC-300/images", "CVC-300/masks")

# CVC-ClinicDB, Kvasir-SEG, ETIS-LARIB
for sub in DATASET_MAP["chakramodel-evaluation-datasets"]["sub_datasets"]:
    run_image_ds(sub["name"], "chakramodel-evaluation-datasets",
                 sub["img_sub"], sub["mask_sub"])

# HyperKvasir
run_image_ds("HyperKvasir-Seg",
    "hyperkvasir-dataset-first-half-and-and-ld-dataset",
    "hyper-kvasir-segmented-images-part 3/segmented-images/images",
    "hyper-kvasir-segmented-images-part 3/segmented-images/masks")

# PolypDB
for sub in DATASET_MAP["polypdb-polyp-raw"]["sub_datasets"]:
    run_image_ds(sub["name"], "polypdb-polyp-raw", sub["img_sub"], sub["mask_sub"])

print("\n" + "=" * 60)
print("VIDEO DATASETS")
print("=" * 60)

run_video_ds("PolypGen2021-Video",    "polypgen20021-video",                  is_neg=False)
run_video_ds("LDPolyp-NoPolyp",      "ldpolypvideowithoutpolyps",            is_neg=True)   # ← fixed
run_video_ds("LDPolyp-PolypOnly",    "ldpolypvideopolyponly",                 is_neg=False)
run_video_ds("HyperKvasir-Video-P2A","hyperkvasir-labeled-videos-part2-001",  is_neg=False)
run_video_ds("HyperKvasir-Video-P2B","hperkvasir-labeled-videos-part2-002",   is_neg=False)
run_video_ds("CVC-Video",            "cvc-sample-video",                      is_neg=False)

print("\n✓ All evaluations complete.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Comprehensive comparison table — v9
# Verdict thresholds: PASS ≥ 0.65  WEAK 0.50–0.65  FAIL < 0.50
# ─────────────────────────────────────────────────────────────

rows = []
for r in ALL_RESULTS:
    if r.get("status") != "done":
        continue

    is_neg  = r.get("is_negative", False)
    ds_type = r.get("type", "unknown")

    if is_neg:
        spec = r["specificity"]
        rows.append({
            "Dataset":      r["name"],
            "Type":         "video-negative",
            "N Samples":    r.get("n_frames_sampled", "-"),
            "Dice Mean":    "N/A (negative ctrl)",
            "Dice Std":     "-",
            "IoU Mean":     "-",
            "Specificity":  f"{spec:.4f}",
            "FP Rate":      f"{r['false_positive_rate']:.4f}",
            "YOLO Detect%": "-",
            "Verdict":      "✓ PASS" if spec >= 0.90 else "✗ FAIL",
        })
    elif "dice_mean" in r:
        n_eval = r.get("n_evaluated", r.get("n_frames_total", 1))
        n_yolo = r.get("yolo_detections", 0)
        yolo_pct = f"{100*n_yolo/n_eval:.0f}%" if n_eval > 0 else "-"
        dm = r["dice_mean"]
        verdict = "✓ PASS" if dm >= 0.65 else ("~ WEAK" if dm >= 0.50 else "✗ FAIL")
        rows.append({
            "Dataset":      r["name"],
            "Type":         ds_type,
            "N Samples":    n_eval,
            "Dice Mean":    f"{dm:.4f}",
            "Dice Std":     f"{r['dice_std']:.4f}",
            "IoU Mean":     f"{r['iou_mean']:.4f}",
            "Specificity":  "-",
            "FP Rate":      "-",
            "YOLO Detect%": yolo_pct,
            "Verdict":      verdict,
        })
    else:
        rows.append({
            "Dataset":      r["name"],
            "Type":         ds_type,
            "N Samples":    r.get("n_frames_sampled", "-"),
            "Dice Mean":    "-",
            "Dice Std":     "-",
            "IoU Mean":     "-",
            "Specificity":  "-",
            "FP Rate":      "-",
            "YOLO Detect%": f"{100*r.get('detection_rate',0):.0f}%",
            "Verdict":      "~ INFO (no GT masks)",
        })

df = pd.DataFrame(rows)
print("\n" + "=" * 90)
print("CHAKRAMODEL v9 — FULL EVALUATION COMPARISON TABLE")
print(f"  (YOLO_CONF_THRESH={YOLO_CONF_THRESH}  |  PASS≥0.65  WEAK≥0.50  FAIL<0.50)")
print("=" * 90)
print(df.to_string(index=False))
print("=" * 90)

image_rows = [r for r in ALL_RESULTS
              if r.get("status")=="done" and "dice_mean" in r and r.get("type")=="image"]
if image_rows:
    all_dice = [r["dice_mean"] for r in image_rows]
    print(f"\nImage datasets summary:")
    print(f"  Overall mean Dice : {np.mean(all_dice):.4f}")
    print(f"  Best dataset      : {max(image_rows, key=lambda x: x['dice_mean'])['name']} ({max(all_dice):.4f})")
    print(f"  Worst dataset     : {min(image_rows, key=lambda x: x['dice_mean'])['name']} ({min(all_dice):.4f})")

df.to_csv(f"{WORKING_DIR}/comparison_table_v9.csv", index=False)
print(f"\nSaved: {WORKING_DIR}/comparison_table_v9.csv")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

image_rows = [r for r in ALL_RESULTS
              if r.get("status") == "done" and "dice_mean" in r]

if image_rows:
    names  = [r["name"].split("/")[-1] for r in image_rows]
    means  = [r["dice_mean"] for r in image_rows]
    stds   = [r["dice_std"]  for r in image_rows]
    colors = ["#2ecc71" if m >= 0.65 else "#e67e22" if m >= 0.50 else "#e74c3c" for m in means]

    fig, ax = plt.subplots(figsize=(max(12, len(names)*0.75), 6))
    ax.bar(names, means, yerr=stds, capsize=4,
           color=colors, alpha=0.85, edgecolor="white", linewidth=0.5)
    ax.axhline(0.65, color="#2ecc71", linestyle="--", linewidth=1.2, label="PASS threshold (0.65)")
    ax.axhline(0.50, color="#e67e22", linestyle="--", linewidth=1.2, label="WEAK threshold (0.50)")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Dice Score")
    ax.set_title(f"ChakraModel v9 — Dice Score per Dataset  (conf≥{YOLO_CONF_THRESH})", fontsize=13)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    p1 = mpatches.Patch(color="#2ecc71", alpha=0.85, label="≥0.65 PASS")
    p2 = mpatches.Patch(color="#e67e22", alpha=0.85, label="0.50–0.65 WEAK")
    p3 = mpatches.Patch(color="#e74c3c", alpha=0.85, label="<0.50 FAIL")
    ax.legend(handles=[p1, p2, p3], loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{WORKING_DIR}/dice_comparison_v9.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Plot saved → dice_comparison_v9.png")

In [ ]:
import json as _json

report = {
    "version":              "v9",
    "yolo_conf_thresh":     YOLO_CONF_THRESH,
    "timestamp":            time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "n_datasets_evaluated": len([r for r in ALL_RESULTS if r.get("status")=="done"]),
    "results":              ALL_RESULTS,
}

out = f"{WORKING_DIR}/combined_report_v9.json"
with open(out, "w") as f:
    _json.dump(report, f, indent=2, default=str)
print(f"Full report saved: {out}")

os.chdir(WORKING_DIR)
os.system("zip -r chakra_v9_artifacts.zip "
          "combined_report_v9.json comparison_table_v9.csv dice_comparison_v9.png "
          "> /dev/null")
print("Done! Download chakra_v9_artifacts.zip from the output panel.")